# 06 — Modèle final ExpenseAI

## Objectif et protocole final

Ce notebook verrouille la configuration issue de `05_model_optimization.ipynb`, sélectionne une éventuelle calibration et un seuil **uniquement sur le train**, entraîne l'artefact final, puis réalise une seule évaluation sur le test de référence.

Configuration immuable : `LogisticRegression(C=1, penalty="l2", class_weight="balanced", max_iter=2000, random_state=42)`. Aucun GridSearch, autre algorithme ou rééchantillonnage n'est introduit.

Le modèle reste une **aide à la décision humaine** : il estime ou signale un risque de refus, sans établir de certitude ni de causalité.

## 1. Imports et reproductibilité

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import inspect
import json
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import shap
from IPython.display import Markdown, display
from sqlalchemy import text

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    fbeta_score,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
pio.templates.default = "plotly_white"

RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from database.connection import create_db_engine

/Users/imanebenamar/Desktop/Projet final/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Chargement de la vue PostgreSQL

In [2]:
engine = create_db_engine()
try:
    with engine.connect() as connection:
        df = pd.read_sql(text("SELECT * FROM v_ml_expenses"), connection)
finally:
    engine.dispose()

numeric_columns_from_sql = [
    "amount_ttc",
    "tax_rate",
    "annee",
    "mois",
    "jour",
    "jour_semaine",
    "trimestre",
    "est_weekend",
    "target",
]
for column in numeric_columns_from_sql:
    df[column] = pd.to_numeric(df[column], errors="coerce")
df["billable"] = df["billable"].astype(int)

target_counts = df["target"].value_counts().sort_index()
data_control = pd.Series(
    {
        "Lignes": len(df),
        "Approuvées": int(target_counts.get(0, 0)),
        "Refusées": int(target_counts.get(1, 0)),
        "Valeurs manquantes": int(df.isna().sum().sum()),
        "Groupes": int(df["expense_group"].nunique()),
    },
    name="Valeur",
)
display(data_control.to_frame())

assert len(df) == 7070
assert target_counts.to_dict() == {0: 6956, 1: 114}
assert set(df["target"].unique()) == {0, 1}
assert df.isna().sum().sum() == 0

,Valeur
Lignes,7070
Approuvées,6956
Refusées,114
Valeurs manquantes,0
Groupes,6253


## 3. Reproduction exacte du split externe de référence

In [3]:
FEATURES = [
    "type",
    "amount_ttc",
    "billable",
    "project_code",
    "tax_rate",
    "annee",
    "mois",
    "jour",
    "jour_semaine",
    "trimestre",
    "est_weekend",
]

X = df[FEATURES].copy()
y = df["target"].astype(int).copy()
groups = df["expense_group"].astype(str).copy()

outer_split = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)
train_indices, test_indices = next(outer_split.split(X, y, groups=groups))

X_train = X.iloc[train_indices].copy()
y_train = y.iloc[train_indices].copy()
groups_train = groups.iloc[train_indices].copy()

# Le jeu de test est matérialisé pour reproduire le split, puis reste gelé.
X_test = X.iloc[test_indices].copy()
y_test = y.iloc[test_indices].copy()
groups_test = groups.iloc[test_indices].copy()

assert set(groups_train).isdisjoint(set(groups_test))
assert len(X_train) == 5656 and int(y_train.sum()) == 91
assert len(X_test) == 1414 and int(y_test.sum()) == 23

split_control = pd.DataFrame(
    {
        "Jeu": ["Train", "Test de référence gelé"],
        "Lignes": [len(train_indices), len(test_indices)],
        "Groupes": [groups_train.nunique(), groups_test.nunique()],
        "Refus": [int(y.iloc[train_indices].sum()), int(y.iloc[test_indices].sum())],
        "Taux de refus (%)": [
            y.iloc[train_indices].mean() * 100,
            y.iloc[test_indices].mean() * 100,
        ],
    }
)
display(split_control)

,Jeu,Lignes,Groupes,Refus,Taux de refus (%)
0,Train,5656,5001,91,1.6089
1,Test de référence gelé,1414,1252,23,1.6266


> **Protocole du test.** Le jeu de test de référence a été gelé pendant toute la phase d'optimisation. Il n'a contribué ni au choix des hyperparamètres, ni à la stratégie de déséquilibre, ni au choix du seuil. Il a déjà été observé dans `04_modeling.ipynb`, mais il n'a pas été utilisé pendant l'optimisation du notebook 05.

## 4. Pipeline de base verrouillé

In [4]:
CATEGORICAL_FEATURES = ["type", "project_code"]
NUMERIC_FEATURES = [
    "amount_ttc",
    "billable",
    "tax_rate",
    "annee",
    "mois",
    "jour",
    "jour_semaine",
    "trimestre",
    "est_weekend",
]


def build_final_base_pipeline() -> Pipeline:
    """Retourne une nouvelle instance du pipeline logistique verrouillé."""
    preprocessing = ColumnTransformer(
        transformers=[
            (
                "categories",
                OneHotEncoder(handle_unknown="ignore"),
                CATEGORICAL_FEATURES,
            ),
            (
                "numeriques",
                Pipeline([("standardisation", StandardScaler())]),
                NUMERIC_FEATURES,
            ),
        ],
        remainder="drop",
    )
    model = LogisticRegression(
        C=1,
        penalty="l2",
        class_weight="balanced",
        max_iter=2000,
        random_state=RANDOM_STATE,
    )
    return Pipeline(
        steps=[
            ("preprocessing", preprocessing),
            ("model", model),
        ]
    )


pipeline_a = build_final_base_pipeline()
pipeline_b = build_final_base_pipeline()
assert pipeline_a is not pipeline_b
assert pipeline_a.get_params()["model__C"] == 1
assert pipeline_a.get_params()["model__class_weight"] == "balanced"

display(
    pd.Series(
        {
            "Algorithme": "Logistic Regression",
            "C": 1,
            "Pénalité": "l2",
            "Poids des classes": "balanced",
            "Features": len(FEATURES),
            "Identifiant de groupe dans les features": "non",
        },
        name="Configuration verrouillée",
    ).to_frame()
)

,Configuration verrouillée
Algorithme,Logistic Regression
C,1
Pénalité,l2
Poids des classes,balanced
Features,11
Identifiant de groupe dans les features,non


## Calibration des probabilités

`class_weight="balanced"` modifie le poids des classes pendant l'apprentissage. Les valeurs de `predict_proba()` restent utiles pour classer les dépenses selon leur risque, mais elles ne doivent pas être interprétées automatiquement comme des probabilités métier parfaitement calibrées.

ExpenseAI devant afficher un résultat compréhensible, trois variantes sont comparées : modèle non calibré, calibration `sigmoid` et calibration `isotonic`. `sigmoid` est a priori la méthode la plus simple et la plus stable avec peu de refus ; `isotonic` est plus flexible et donc plus exposée au surapprentissage.

La sélection est réalisée uniquement avec des probabilités out-of-fold du train. La calibration interne et son évaluation externe respectent toujours `expense_group`.

### Règle de sélection définie avant les résultats

- `sigmoid` est admissible si elle améliore le Brier d'au moins 2 % **ou** la Log Loss d'au moins 5 %, maintient l'ECE à `+0,005` près et ne perd pas plus de `0,01` de PR-AUC.
- `isotonic` n'est préférée que si elle améliore à son tour Brier et Log Loss d'au moins 2 %, maintient l'ECE et la PR-AUC, et gagne le Brier dans au moins 4 folds externes sur 5.
- sinon, la méthode la plus simple admissible est conservée ; en l'absence d'amélioration réelle, le modèle reste non calibré et son résultat est nommé **score de risque**.

L'ECE complète Brier et Log Loss, mais n'est jamais utilisée seule.

In [5]:
CALIBRATION_METHODS = ["non calibré", "sigmoid", "isotonic"]
outer_calibration_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

oof_probabilities_by_method = {
    method: np.full(len(X_train), np.nan, dtype=float)
    for method in CALIBRATION_METHODS
}
calibration_fold_rows = []

for outer_fold, (outer_train_idx, outer_validation_idx) in enumerate(
    outer_calibration_cv.split(X_train, y_train, groups=groups_train),
    start=1,
):
    X_outer_train = X_train.iloc[outer_train_idx]
    y_outer_train = y_train.iloc[outer_train_idx]
    groups_outer_train = groups_train.iloc[outer_train_idx]
    X_outer_validation = X_train.iloc[outer_validation_idx]
    y_outer_validation = y_train.iloc[outer_validation_idx]
    groups_outer_validation = groups_train.iloc[outer_validation_idx]

    assert set(groups_outer_train).isdisjoint(set(groups_outer_validation))

    base_fold_model = build_final_base_pipeline()
    base_fold_model.fit(X_outer_train, y_outer_train)
    base_probabilities = base_fold_model.predict_proba(X_outer_validation)[:, 1]
    oof_probabilities_by_method["non calibré"][outer_validation_idx] = (
        base_probabilities
    )

    calibration_fold_rows.append(
        {
            "Fold": outer_fold,
            "Méthode": "non calibré",
            "Brier Score": brier_score_loss(y_outer_validation, base_probabilities),
            "Log Loss": log_loss(
                y_outer_validation, base_probabilities, labels=[0, 1]
            ),
        }
    )

    inner_calibration_cv = StratifiedGroupKFold(
        n_splits=3,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    inner_splits = list(
        inner_calibration_cv.split(
            X_outer_train,
            y_outer_train,
            groups=groups_outer_train,
        )
    )
    for inner_train_idx, inner_validation_idx in inner_splits:
        inner_train_groups = groups_outer_train.iloc[inner_train_idx]
        inner_validation_groups = groups_outer_train.iloc[inner_validation_idx]
        assert set(inner_train_groups).isdisjoint(set(inner_validation_groups))

    for method in ["sigmoid", "isotonic"]:
        calibrated_fold_model = CalibratedClassifierCV(
            estimator=build_final_base_pipeline(),
            method=method,
            cv=inner_splits,
            ensemble=False,
        )
        calibrated_fold_model.fit(X_outer_train, y_outer_train)
        calibrated_probabilities = calibrated_fold_model.predict_proba(
            X_outer_validation
        )[:, 1]
        oof_probabilities_by_method[method][outer_validation_idx] = (
            calibrated_probabilities
        )
        calibration_fold_rows.append(
            {
                "Fold": outer_fold,
                "Méthode": method,
                "Brier Score": brier_score_loss(
                    y_outer_validation, calibrated_probabilities
                ),
                "Log Loss": log_loss(
                    y_outer_validation, calibrated_probabilities, labels=[0, 1]
                ),
            }
        )

for method, probabilities in oof_probabilities_by_method.items():
    assert np.isfinite(probabilities).all(), f"Probabilités OOF incomplètes : {method}"

calibration_fold_metrics = pd.DataFrame(calibration_fold_rows)
display(calibration_fold_metrics)

,Fold,Méthode,Brier Score,Log Loss
0,1,non calibré,0.1355,0.4188
1,1,sigmoid,0.0155,0.0815
2,1,isotonic,0.0153,0.1568
3,2,non calibré,0.1240,0.3832
4,2,sigmoid,0.0151,0.0706
5,2,isotonic,0.0151,0.0718
6,3,non calibré,0.1235,0.3906
7,3,sigmoid,0.0153,0.0718
8,3,isotonic,0.0156,0.0742
9,4,non calibré,0.1135,0.3592


## 5. Métriques de calibration out-of-fold

In [6]:
def expected_calibration_error(
    y_true: pd.Series | np.ndarray,
    probabilities: np.ndarray,
    n_bins: int = 8,
) -> float:
    """Estime l'ECE avec des bins quantiles, mieux peuplés pour une cible rare."""
    calibration_data = pd.DataFrame(
        {
            "y": np.asarray(y_true, dtype=float),
            "probability": np.asarray(probabilities, dtype=float),
        }
    )
    calibration_data["bin"] = pd.qcut(
        calibration_data["probability"],
        q=n_bins,
        labels=False,
        duplicates="drop",
    )
    ece = 0.0
    for _, bin_data in calibration_data.groupby("bin", observed=True):
        weight = len(bin_data) / len(calibration_data)
        ece += weight * abs(bin_data["y"].mean() - bin_data["probability"].mean())
    return float(ece)


calibration_rows = []
for method, probabilities in oof_probabilities_by_method.items():
    calibration_rows.append(
        {
            "Méthode": method,
            "Brier Score": brier_score_loss(y_train, probabilities),
            "Log Loss": log_loss(y_train, probabilities, labels=[0, 1]),
            "ECE": expected_calibration_error(y_train, probabilities, n_bins=8),
            "ROC-AUC": roc_auc_score(y_train, probabilities),
            "PR-AUC": average_precision_score(y_train, probabilities),
        }
    )

calibration_summary = pd.DataFrame(calibration_rows).set_index("Méthode")
display(calibration_summary)
display(
    Markdown(
        "**Lecture :** Brier Score, Log Loss et ECE sont meilleurs lorsqu'ils "
        "diminuent. ROC-AUC et PR-AUC sont meilleurs lorsqu'ils augmentent."
    )
)

,Brier Score,Log Loss,ECE,ROC-AUC,PR-AUC
Méthode,,,,,
non calibré,0.1286,0.3992,0.2327,0.7770,0.0864
sigmoid,0.0154,0.0738,0.0055,0.7735,0.0812
isotonic,0.0153,0.0888,0.0032,0.7646,0.0782


**Lecture :** Brier Score, Log Loss et ECE sont meilleurs lorsqu'ils diminuent. ROC-AUC et PR-AUC sont meilleurs lorsqu'ils augmentent.

In [7]:
calibration_figure = go.Figure()
calibration_figure.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        name="Calibration idéale",
        line={"color": "#64748B", "dash": "dash"},
    )
)

method_colors = {
    "non calibré": "#DC2626",
    "sigmoid": "#2563EB",
    "isotonic": "#0F766E",
}
for method, probabilities in oof_probabilities_by_method.items():
    observed_frequency, predicted_probability = calibration_curve(
        y_train,
        probabilities,
        n_bins=8,
        strategy="quantile",
    )
    calibration_figure.add_trace(
        go.Scatter(
            x=predicted_probability,
            y=observed_frequency,
            mode="lines+markers",
            name=method,
            line={"color": method_colors[method]},
        )
    )

calibration_figure.update_layout(
    title="Calibration out-of-fold sur le train — 8 bins quantiles",
    xaxis_title="Probabilité moyenne prédite",
    yaxis_title="Fréquence observée de refus",
    xaxis={"range": [0, 1]},
    yaxis={"range": [0, 1]},
)
calibration_figure.show()

## 6. Choix de la calibration — train uniquement

In [8]:
SIGMOID_MIN_BRIER_RELATIVE_GAIN = 0.02
SIGMOID_MIN_LOGLOSS_RELATIVE_GAIN = 0.05
MAX_ECE_DEGRADATION = 0.005
MAX_PR_AUC_DEGRADATION = 0.01
ISOTONIC_MIN_RELATIVE_GAIN = 0.02
ISOTONIC_MIN_WINNING_FOLDS = 4

base_metrics = calibration_summary.loc["non calibré"]
sigmoid_metrics = calibration_summary.loc["sigmoid"]
isotonic_metrics = calibration_summary.loc["isotonic"]

sigmoid_brier_gain = 1 - sigmoid_metrics["Brier Score"] / base_metrics["Brier Score"]
sigmoid_logloss_gain = 1 - sigmoid_metrics["Log Loss"] / base_metrics["Log Loss"]
sigmoid_eligible = (
    (
        sigmoid_brier_gain >= SIGMOID_MIN_BRIER_RELATIVE_GAIN
        or sigmoid_logloss_gain >= SIGMOID_MIN_LOGLOSS_RELATIVE_GAIN
    )
    and sigmoid_metrics["ECE"] <= base_metrics["ECE"] + MAX_ECE_DEGRADATION
    and sigmoid_metrics["PR-AUC"] >= base_metrics["PR-AUC"] - MAX_PR_AUC_DEGRADATION
)

sigmoid_fold_brier = calibration_fold_metrics[
    calibration_fold_metrics["Méthode"].eq("sigmoid")
].sort_values("Fold")["Brier Score"].to_numpy()
isotonic_fold_brier = calibration_fold_metrics[
    calibration_fold_metrics["Méthode"].eq("isotonic")
].sort_values("Fold")["Brier Score"].to_numpy()
isotonic_winning_folds = int((isotonic_fold_brier < sigmoid_fold_brier).sum())

isotonic_brier_gain_vs_sigmoid = (
    1 - isotonic_metrics["Brier Score"] / sigmoid_metrics["Brier Score"]
)
isotonic_logloss_gain_vs_sigmoid = (
    1 - isotonic_metrics["Log Loss"] / sigmoid_metrics["Log Loss"]
)
isotonic_preferred_to_sigmoid = (
    isotonic_brier_gain_vs_sigmoid >= ISOTONIC_MIN_RELATIVE_GAIN
    and isotonic_logloss_gain_vs_sigmoid >= ISOTONIC_MIN_RELATIVE_GAIN
    and isotonic_metrics["ECE"] <= sigmoid_metrics["ECE"] + MAX_ECE_DEGRADATION
    and isotonic_metrics["PR-AUC"]
    >= sigmoid_metrics["PR-AUC"] - MAX_PR_AUC_DEGRADATION
    and isotonic_winning_folds >= ISOTONIC_MIN_WINNING_FOLDS
)

base_fold_brier = calibration_fold_metrics[
    calibration_fold_metrics["Méthode"].eq("non calibré")
].sort_values("Fold")["Brier Score"].to_numpy()
isotonic_wins_vs_base = int((isotonic_fold_brier < base_fold_brier).sum())
isotonic_directly_eligible = (
    not sigmoid_eligible
    and isotonic_metrics["Brier Score"] <= base_metrics["Brier Score"] * 0.98
    and isotonic_metrics["Log Loss"] <= base_metrics["Log Loss"] * 0.95
    and isotonic_metrics["ECE"] <= base_metrics["ECE"] + MAX_ECE_DEGRADATION
    and isotonic_metrics["PR-AUC"] >= base_metrics["PR-AUC"] - MAX_PR_AUC_DEGRADATION
    and isotonic_wins_vs_base >= ISOTONIC_MIN_WINNING_FOLDS
)

if sigmoid_eligible:
    SELECTED_CALIBRATION = "sigmoid"
    calibration_reason = (
        "Sigmoid améliore clairement au moins une métrique probabiliste, "
        "maintient l'ECE et la PR-AUC, et reste la solution la plus simple."
    )
    if isotonic_preferred_to_sigmoid:
        SELECTED_CALIBRATION = "isotonic"
        calibration_reason = (
            "Isotonic améliore suffisamment Brier et Log Loss face à sigmoid, "
            "sans dégrader les autres critères, et son gain est stable par fold."
        )
elif isotonic_directly_eligible:
    SELECTED_CALIBRATION = "isotonic"
    calibration_reason = (
        "Sigmoid n'est pas admissible, tandis qu'isotonic améliore clairement "
        "et de façon stable le modèle non calibré."
    )
else:
    SELECTED_CALIBRATION = None
    calibration_reason = (
        "Aucune calibration n'améliore suffisamment les résultats selon la règle "
        "préétablie ; la sortie sera présentée comme un score de risque."
    )

selected_method_label = SELECTED_CALIBRATION or "non calibré"
selected_oof_probabilities = oof_probabilities_by_method[selected_method_label]

display(
    pd.Series(
        {
            "Gain relatif Brier sigmoid": sigmoid_brier_gain,
            "Gain relatif Log Loss sigmoid": sigmoid_logloss_gain,
            "Sigmoid admissible": sigmoid_eligible,
            "Folds Brier gagnés par isotonic face à sigmoid": (
                f"{isotonic_winning_folds}/5"
            ),
            "Isotonic préférable à sigmoid": isotonic_preferred_to_sigmoid,
            "Calibration retenue": selected_method_label,
            "Justification": calibration_reason,
        },
        name="Décision de calibration",
    ).to_frame()
)

,Décision de calibration
Gain relatif Brier sigmoid,0.8805
Gain relatif Log Loss sigmoid,0.8151
Sigmoid admissible,True
Folds Brier gagnés par isotonic face à sigmoid,3/5
Isotonic préférable à sigmoid,False
Calibration retenue,sigmoid
Justification,Sigmoid améliore clairement au moins une métri...


## 7. Recalcul des seuils après calibration

Les seuils `0,50`, `0,87` et `0,34` du notebook 05 concernaient les scores non calibrés. Ils ne sont pas réutilisés. Les seuils ci-dessous sont recalculés avec les nouvelles probabilités out-of-fold de la variante retenue, sur le train uniquement.

In [9]:
def calculate_threshold_metrics(probabilities: np.ndarray) -> pd.DataFrame:
    """Calcule les métriques de décision pour les seuils de 0,01 à 0,99."""
    rows = []
    for threshold in np.arange(0.01, 1.00, 0.01):
        predictions = (probabilities >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_train, predictions, labels=[0, 1]).ravel()
        rows.append(
            {
                "Seuil": float(np.round(threshold, 2)),
                "Precision": precision_score(y_train, predictions, zero_division=0),
                "Recall": recall_score(y_train, predictions, zero_division=0),
                "F1": f1_score(y_train, predictions, zero_division=0),
                "F2": fbeta_score(y_train, predictions, beta=2, zero_division=0),
                "Balanced Accuracy": balanced_accuracy_score(y_train, predictions),
                "FP": int(fp),
                "FN": int(fn),
                "TP": int(tp),
                "TN": int(tn),
            }
        )
    return pd.DataFrame(rows)


threshold_metrics = calculate_threshold_metrics(selected_oof_probabilities)
display(threshold_metrics.head())

,Seuil,Precision,Recall,F1,F2,Balanced Accuracy,FP,FN,TP,TN
0,0.0100,0.0259,0.8352,0.0503,0.1154,0.6612,2853,15,76,2712
1,0.0200,0.0428,0.6923,0.0806,0.1715,0.7195,1410,28,63,4155
2,0.0300,0.0588,0.5495,0.1063,0.2059,0.7028,800,41,50,4765
3,0.0400,0.0904,0.4945,0.1528,0.2610,0.7066,453,46,45,5112
4,0.0500,0.1070,0.3846,0.1675,0.2533,0.6661,292,56,35,5273


In [10]:
threshold_figure = go.Figure()
for metric, color in [
    ("Precision", "#0F766E"),
    ("Recall", "#DC2626"),
    ("F1", "#2563EB"),
    ("F2", "#7C3AED"),
]:
    threshold_figure.add_trace(
        go.Scatter(
            x=threshold_metrics["Seuil"],
            y=threshold_metrics[metric],
            mode="lines",
            name=metric,
            line={"color": color},
        )
    )
threshold_figure.update_layout(
    title=f"Métriques OOF selon le seuil — calibration {selected_method_label}",
    xaxis_title="Seuil",
    yaxis_title="Score",
    hovermode="x unified",
)
threshold_figure.show()

## 8. Seuils candidats et verrouillage du seuil principal

In [11]:
scenario_f1 = threshold_metrics.sort_values(
    ["F1", "Precision", "Seuil"],
    ascending=[False, False, False],
).iloc[0]
scenario_f2 = threshold_metrics.sort_values(
    ["F2", "Recall", "Precision", "Seuil"],
    ascending=[False, False, False, False],
).iloc[0]
strong_detection_options = threshold_metrics[threshold_metrics["Recall"] >= 0.70]
if strong_detection_options.empty:
    scenario_strong_detection = None
else:
    scenario_strong_detection = strong_detection_options.sort_values(
        ["Precision", "F2", "Seuil"],
        ascending=[False, False, False],
    ).iloc[0]

scenario_rows = []
for scenario_name, scenario in [
    ("Seuil F1", scenario_f1),
    ("Seuil F2 — principal", scenario_f2),
    ("Forte détection — recall ≥ 70 %", scenario_strong_detection),
]:
    if scenario is not None:
        values = scenario.to_dict()
        values["Scénario"] = scenario_name
        scenario_rows.append(values)

threshold_candidates = pd.DataFrame(scenario_rows)[
    [
        "Scénario",
        "Seuil",
        "Precision",
        "Recall",
        "F1",
        "F2",
        "Balanced Accuracy",
        "FP",
        "FN",
    ]
]

# En l'absence de coût métier chiffré, F2 privilégie la détection sans maximiser
# le recall à tout prix. Ce choix est effectué avant toute évaluation du test.
PRODUCTION_THRESHOLD = float(scenario_f2["Seuil"])
MODEL_LOCKED = True

display(threshold_candidates)
display(
    pd.Series(
        {
            "Algorithme": "Logistic Regression",
            "Hyperparamètres": {
                "C": 1,
                "penalty": "l2",
                "class_weight": "balanced",
                "max_iter": 2000,
                "random_state": RANDOM_STATE,
            },
            "Calibration": selected_method_label,
            "Seuil principal de production": PRODUCTION_THRESHOLD,
            "Critère du seuil": "F2 maximal sur probabilités OOF train",
            "MODEL_LOCKED": MODEL_LOCKED,
        },
        name="Verrouillage final avant test",
    ).to_frame()
)
print(f"Seuil principal de production = {PRODUCTION_THRESHOLD:.2f}")
print("MODEL_LOCKED =", MODEL_LOCKED)

,Scénario,Seuil,Precision,Recall,F1,F2,Balanced Accuracy,FP,FN
0,Seuil F1,0.0700,0.1611,0.2637,0.2000,0.2339,0.6206,125.0000,67.0000
1,Seuil F2 — principal,0.0400,0.0904,0.4945,0.1528,0.2610,0.7066,453.0000,46.0000
2,Forte détection — recall ≥ 70 %,0.0100,0.0259,0.8352,0.0503,0.1154,0.6612,"2,853.0000",15.0000


,Verrouillage final avant test
Algorithme,Logistic Regression
Hyperparamètres,"{'C': 1, 'penalty': 'l2', 'class_weight': 'bal..."
Calibration,sigmoid
Seuil principal de production,0.0400
Critère du seuil,F2 maximal sur probabilités OOF train
MODEL_LOCKED,True


Seuil principal de production = 0.04
MODEL_LOCKED = True


## 9. Entraînement final sur l'intégralité du train

In [12]:
if SELECTED_CALIBRATION is None:
    final_estimator = build_final_base_pipeline()
    final_estimator.fit(X_train, y_train)
    final_training_strategy = "Pipeline de base entraîné sur tout le train"
else:
    final_calibration_cv = list(
        StratifiedGroupKFold(
            n_splits=5,
            shuffle=True,
            random_state=RANDOM_STATE,
        ).split(
            X_train,
            y_train,
            groups=groups_train,
        )
    )
    for calibration_train_idx, calibration_validation_idx in final_calibration_cv:
        assert set(groups_train.iloc[calibration_train_idx]).isdisjoint(
            set(groups_train.iloc[calibration_validation_idx])
        )

    calibrated_signature = inspect.signature(CalibratedClassifierCV)
    if "ensemble" not in calibrated_signature.parameters:
        raise RuntimeError(
            "La version installée de scikit-learn ne supporte pas ensemble=False. "
            "L'entraînement est interrompu plutôt que d'utiliser une alternative silencieuse."
        )

    final_estimator = CalibratedClassifierCV(
        estimator=build_final_base_pipeline(),
        method=SELECTED_CALIBRATION,
        cv=final_calibration_cv,
        ensemble=False,
    )
    final_estimator.fit(X_train, y_train)
    final_training_strategy = (
        "CalibratedClassifierCV avec folds groupés pré-calculés et ensemble=False ; "
        "le modèle de base final est entraîné sur tout le train."
    )

display(
    pd.Series(
        {
            "Lignes d'entraînement": len(X_train),
            "Refus dans l'entraînement": int(y_train.sum()),
            "Calibration": selected_method_label,
            "Stratégie": final_training_strategy,
        },
        name="Entraînement final",
    ).to_frame()
)

,Entraînement final
Lignes d'entraînement,5656
Refus dans l'entraînement,91
Calibration,sigmoid
Stratégie,CalibratedClassifierCV avec folds groupés pré-...


## Évaluation finale sur le jeu de test

`MODEL_LOCKED = True` est vérifié avant la première prédiction. À partir de cette frontière, l'algorithme, les hyperparamètres, la calibration et le seuil ne sont plus modifiables. Le test sert uniquement à mesurer la généralisation ; ses résultats ne déclenchent aucune nouvelle décision de modélisation.

In [13]:
assert MODEL_LOCKED
print("MODEL_LOCKED =", MODEL_LOCKED)

# Ouverture unique du test : les probabilités sont calculées une fois puis réutilisées.
test_probabilities = final_estimator.predict_proba(X_test)[:, 1]
test_predictions = (test_probabilities >= PRODUCTION_THRESHOLD).astype(int)

test_metrics = {
    "accuracy": accuracy_score(y_test, test_predictions),
    "balanced_accuracy": balanced_accuracy_score(y_test, test_predictions),
    "precision_refusee": precision_score(y_test, test_predictions, zero_division=0),
    "recall_refusee": recall_score(y_test, test_predictions, zero_division=0),
    "f1_refusee": f1_score(y_test, test_predictions, zero_division=0),
    "f2_refusee": fbeta_score(
        y_test, test_predictions, beta=2, zero_division=0
    ),
    "roc_auc": roc_auc_score(y_test, test_probabilities),
    "average_precision": average_precision_score(y_test, test_probabilities),
    "brier_score": brier_score_loss(y_test, test_probabilities),
    "log_loss": log_loss(y_test, test_probabilities, labels=[0, 1]),
}
display(
    pd.Series(test_metrics, name="Valeur").rename_axis("Métrique").to_frame()
)

print(
    classification_report(
        y_test,
        test_predictions,
        labels=[0, 1],
        target_names=["Approuvée", "Refusée"],
        digits=4,
        zero_division=0,
    )
)

MODEL_LOCKED = True


,Valeur
Métrique,
accuracy,0.9074
balanced_accuracy,0.6964
precision_refusee,0.0846
recall_refusee,0.4783
f1_refusee,0.1438
f2_refusee,0.2477
roc_auc,0.8535
average_precision,0.0858
brier_score,0.0159


              precision    recall  f1-score   support

   Approuvée     0.9907    0.9145    0.9510      1391
     Refusée     0.0846    0.4783    0.1438        23

    accuracy                         0.9074      1414
   macro avg     0.5376    0.6964    0.5474      1414
weighted avg     0.9759    0.9074    0.9379      1414



## 10. Matrice de confusion finale

In [14]:
final_confusion = confusion_matrix(y_test, test_predictions, labels=[0, 1])
tn, fp, fn, tp = final_confusion.ravel()

confusion_table = pd.DataFrame(
    final_confusion,
    index=["Réelle Approuvée", "Réelle Refusée"],
    columns=["Prédite Approuvée", "Prédite Refusée"],
)
display(confusion_table)

confusion_figure = go.Figure(
    data=go.Heatmap(
        z=final_confusion,
        x=["Prédite Approuvée", "Prédite Refusée"],
        y=["Réelle Approuvée", "Réelle Refusée"],
        text=final_confusion,
        texttemplate="%{text}",
        colorscale="Blues",
        showscale=False,
    )
)
confusion_figure.update_layout(
    title=f"Matrice de confusion finale — seuil {PRODUCTION_THRESHOLD:.2f}",
    xaxis_title="Prédiction",
    yaxis_title="Réalité",
)
confusion_figure.show()

display(
    Markdown(
        f"- **FP = {fp}** : dépenses approuvées mais signalées pour contrôle.\n"
        f"- **FN = {fn}** : dépenses réellement refusées mais non détectées.\n\n"
        "ExpenseAI signale des lignes à examiner ; la décision reste humaine."
    )
)

,Prédite Approuvée,Prédite Refusée
Réelle Approuvée,1272,119
Réelle Refusée,12,11


- **FP = 119** : dépenses approuvées mais signalées pour contrôle.
- **FN = 12** : dépenses réellement refusées mais non détectées.

ExpenseAI signale des lignes à examiner ; la décision reste humaine.

## 11. Comparaison avec le DummyClassifier

In [15]:
dummy_model = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
dummy_model.fit(X_train, y_train)
dummy_predictions = dummy_model.predict(X_test)
dummy_probabilities = dummy_model.predict_proba(X_test)[:, 1]

dummy_comparison = pd.DataFrame(
    [
        {
            "Modèle": "Dummy — toujours Approuvée",
            "Balanced Accuracy": balanced_accuracy_score(y_test, dummy_predictions),
            "Recall Refusée": recall_score(
                y_test, dummy_predictions, zero_division=0
            ),
            "F1 Refusée": f1_score(y_test, dummy_predictions, zero_division=0),
            "PR-AUC": average_precision_score(y_test, dummy_probabilities),
        },
        {
            "Modèle": "ExpenseAI final",
            "Balanced Accuracy": test_metrics["balanced_accuracy"],
            "Recall Refusée": test_metrics["recall_refusee"],
            "F1 Refusée": test_metrics["f1_refusee"],
            "PR-AUC": test_metrics["average_precision"],
        },
    ]
)
display(dummy_comparison)

,Modèle,Balanced Accuracy,Recall Refusée,F1 Refusée,PR-AUC
0,Dummy — toujours Approuvée,0.5000,0.0000,0.0000,0.0163
1,ExpenseAI final,0.6964,0.4783,0.1438,0.0858


## 12. Courbes finales — mesures descriptives

In [16]:
false_positive_rate, true_positive_rate, _ = roc_curve(y_test, test_probabilities)
roc_figure = go.Figure()
roc_figure.add_trace(
    go.Scatter(
        x=false_positive_rate,
        y=true_positive_rate,
        mode="lines",
        name=f"ExpenseAI — ROC-AUC {test_metrics['roc_auc']:.3f}",
        line={"color": "#2563EB"},
    )
)
roc_figure.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        name="Aléatoire",
        line={"color": "#64748B", "dash": "dash"},
    )
)
roc_figure.update_layout(
    title="Courbe ROC finale",
    xaxis_title="Taux de faux positifs",
    yaxis_title="Taux de vrais positifs",
)
roc_figure.show()

test_precision_curve, test_recall_curve, _ = precision_recall_curve(
    y_test, test_probabilities
)
test_prevalence = float(y_test.mean())
pr_figure = go.Figure()
pr_figure.add_trace(
    go.Scatter(
        x=test_recall_curve,
        y=test_precision_curve,
        mode="lines",
        name=f"ExpenseAI — PR-AUC {test_metrics['average_precision']:.3f}",
        line={"color": "#7C3AED"},
    )
)
pr_figure.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[test_prevalence, test_prevalence],
        mode="lines",
        name=f"Prévalence — {test_prevalence:.3f}",
        line={"color": "#64748B", "dash": "dash"},
    )
)
pr_figure.update_layout(
    title="Courbe Precision-Recall finale — prioritaire pour la cible rare",
    xaxis_title="Recall Refusée",
    yaxis_title="Precision Refusée",
)
pr_figure.show()

test_observed_frequency, test_predicted_probability = calibration_curve(
    y_test,
    test_probabilities,
    n_bins=5,
    strategy="quantile",
)
test_calibration_figure = go.Figure()
test_calibration_figure.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        name="Calibration idéale",
        line={"color": "#64748B", "dash": "dash"},
    )
)
test_calibration_figure.add_trace(
    go.Scatter(
        x=test_predicted_probability,
        y=test_observed_frequency,
        mode="lines+markers",
        name="ExpenseAI final",
        line={"color": "#0F766E"},
    )
)
test_calibration_figure.update_layout(
    title="Calibration finale sur le test — mesure descriptive uniquement",
    xaxis_title="Probabilité moyenne prédite",
    yaxis_title="Fréquence observée de refus",
)
test_calibration_figure.show()

## Limites de l'évaluation finale

- Le test ne contient que **23 refus** : quelques observations peuvent modifier fortement recall et precision.
- La période historique couvre environ une année et les comportements futurs peuvent évoluer.
- Les performances et la calibration devront être surveillées après mise en production.
- Cette évaluation unique décrit la généralisation observée ; elle ne permet pas de qualifier le modèle d'« excellent » ni de garantir ses performances futures.

## 13. Interprétation SHAP du modèle logistique

> La calibration transforme le score produit par le modèle en probabilité mieux calibrée, mais n'altère pas l'interprétation structurelle des variables du modèle de base. Les valeurs SHAP présentées expliquent donc la décision du modèle logistique avant calibration.

Une copie indépendante du pipeline logistique verrouillé est entraînée sur tout le train. Aucune couche de calibration n'est expliquée par SHAP.

In [17]:
shap_base_pipeline = build_final_base_pipeline()
shap_base_pipeline.fit(X_train, y_train)

preprocessor = shap_base_pipeline.named_steps["preprocessing"]
logistic_model = shap_base_pipeline.named_steps["model"]
raw_feature_names = np.asarray(preprocessor.get_feature_names_out(), dtype=str)

# Les codes projet réels sont masqués uniquement dans les sorties d'interprétation.
# Le modèle conserve naturellement ses noms internes inchangés.
safe_feature_names = []
project_modality_number = 0
for feature_name in raw_feature_names:
    if (
        feature_name.startswith("categories__project_code_")
        and not feature_name.endswith("SANS_PROJET")
    ):
        project_modality_number += 1
        safe_feature_names.append(
            f"categories__project_code_MODALITE_{project_modality_number:03d}"
        )
    else:
        safe_feature_names.append(feature_name)
feature_names = np.asarray(safe_feature_names, dtype=str)

background_size = min(500, len(X_train))
explanation_size = min(500, len(X_train))
background_sample = X_train.sample(
    n=background_size,
    random_state=RANDOM_STATE,
)
explanation_sample = X_train.sample(
    n=explanation_size,
    random_state=RANDOM_STATE + 1,
)
background_transformed = preprocessor.transform(background_sample)
explanation_transformed = preprocessor.transform(explanation_sample)

background_masker = shap.maskers.Independent(
    background_transformed,
    max_samples=background_size,
)
linear_explainer = shap.LinearExplainer(logistic_model, background_masker)
shap_explanation = linear_explainer(explanation_transformed)
shap_values = np.asarray(shap_explanation.values)

assert shap_values.shape[1] == len(feature_names)
print("Lignes expliquées :", shap_values.shape[0])
print("Features après transformation :", shap_values.shape[1])

Lignes expliquées : 500
Features après transformation : 223


### Importance SHAP globale

In [18]:
shap_global_importance = (
    pd.DataFrame(
        {
            "Feature": feature_names,
            "Importance SHAP moyenne absolue": np.abs(shap_values).mean(axis=0),
        }
    )
    .sort_values("Importance SHAP moyenne absolue", ascending=False)
    .reset_index(drop=True)
)
display(shap_global_importance.head(20))

top_shap = shap_global_importance.head(15).sort_values(
    "Importance SHAP moyenne absolue"
)
shap_figure = go.Figure(
    go.Bar(
        x=top_shap["Importance SHAP moyenne absolue"],
        y=top_shap["Feature"],
        orientation="h",
        marker_color="#7C3AED",
    )
)
shap_figure.update_layout(
    title="Top 15 des importances SHAP globales",
    xaxis_title="Moyenne de |valeur SHAP| sur le logit",
    yaxis_title="Feature transformée",
)
shap_figure.show()

,Feature,Importance SHAP moyenne absolue
0,categories__type_Frais kilométriques,1.0770
1,numeriques__mois,0.5610
2,categories__project_code_SANS_PROJET,0.5569
3,numeriques__annee,0.3879
4,categories__type_Parking,0.3027
5,numeriques__tax_rate,0.2994
6,numeriques__trimestre,0.2725
7,numeriques__jour_semaine,0.2039
8,categories__type_Carburant,0.1621
9,categories__type_Abonnement Professionnel,0.1523


### Coefficients et odds ratios

Les coefficients décrivent des **associations apprises par le modèle** et ne prouvent aucune causalité. Un coefficient positif pousse le score logistique vers `Refusée`, tandis qu'un coefficient négatif le pousse vers `Approuvée`, toutes choses égales dans la représentation transformée.

In [19]:
coefficient_table = pd.DataFrame(
    {
        "Feature": feature_names,
        "Coefficient": logistic_model.coef_[0],
    }
)
coefficient_table["Odds ratio"] = np.exp(coefficient_table["Coefficient"])

positive_coefficients = coefficient_table.nlargest(15, "Coefficient")
negative_coefficients = coefficient_table.nsmallest(15, "Coefficient")

display(Markdown("#### Associations les plus positives vers `Refusée`"))
display(positive_coefficients)
display(Markdown("#### Associations les plus négatives vers `Approuvée`"))
display(negative_coefficients)

#### Associations les plus positives vers `Refusée`

,Feature,Coefficient,Odds ratio
208,categories__project_code_MODALITE_175,6.9390,"1,031.7876"
197,categories__project_code_MODALITE_164,5.0203,151.4515
177,categories__project_code_MODALITE_144,4.6697,106.6613
42,categories__project_code_MODALITE_010,3.9682,52.8900
107,categories__project_code_MODALITE_075,3.7753,43.6095
211,categories__project_code_MODALITE_178,3.6503,38.4862
161,categories__project_code_MODALITE_129,3.6235,37.4693
13,categories__type_Equipement Télétravail,3.4223,30.6402
108,categories__project_code_MODALITE_076,3.1502,23.3399
24,categories__type_Petit équipement/petit matériel,2.6574,14.2589


#### Associations les plus négatives vers `Approuvée`

,Feature,Coefficient,Odds ratio
17,categories__type_Frais kilométriques,-4.2104,0.0148
27,categories__type_Repas à emporter,-2.8379,0.0585
8,categories__type_Carburant,-2.5303,0.0796
23,categories__type_Parking,-2.0103,0.1339
148,categories__project_code_MODALITE_116,-1.6760,0.1871
30,categories__type_Train,-1.6090,0.2001
32,categories__type_Vaccin,-1.3566,0.2575
181,categories__project_code_MODALITE_148,-1.3450,0.2605
149,categories__project_code_MODALITE_117,-1.1777,0.3080
150,categories__project_code_MODALITE_118,-1.1641,0.3122


### Explication locale sur une dépense synthétique

In [20]:
synthetic_expense = pd.DataFrame(
    [
        {
            "type": "TYPE_SYNTHETIQUE_INCONNU",
            "amount_ttc": 125.50,
            "billable": 0,
            "project_code": "PROJET_SYNTHETIQUE_INCONNU",
            "tax_rate": 0.20,
            "annee": 2026,
            "mois": 6,
            "jour": 15,
            "jour_semaine": 0,
            "trimestre": 2,
            "est_weekend": 0,
        }
    ]
)

synthetic_transformed = preprocessor.transform(synthetic_expense)
synthetic_shap = np.asarray(linear_explainer(synthetic_transformed).values)[0]
local_contributions = pd.DataFrame(
    {
        "Feature": feature_names,
        "Contribution SHAP": synthetic_shap,
    }
)
local_positive = local_contributions.nlargest(5, "Contribution SHAP")
local_negative = local_contributions.nsmallest(5, "Contribution SHAP")

synthetic_score = float(final_estimator.predict_proba(synthetic_expense)[:, 1][0])
synthetic_target = int(synthetic_score >= PRODUCTION_THRESHOLD)
synthetic_label = "Refusée" if synthetic_target == 1 else "Approuvée"
score_display_name = (
    "Probabilité calibrée de refus"
    if SELECTED_CALIBRATION is not None
    else "Score de risque de refus"
)

display(
    pd.Series(
        {
            "Prédiction estimée": synthetic_label,
            score_display_name: synthetic_score,
            "Seuil appliqué": PRODUCTION_THRESHOLD,
        },
        name="Démonstration synthétique",
    ).to_frame()
)
display(Markdown("#### Principaux facteurs poussant vers `Refusée`"))
display(local_positive)
display(Markdown("#### Principaux facteurs poussant vers `Approuvée`"))
display(local_negative)

,Démonstration synthétique
Prédiction estimée,Approuvée
Probabilité calibrée de refus,0.0044
Seuil appliqué,0.0400


#### Principaux facteurs poussant vers `Refusée`

,Feature,Contribution SHAP
17,categories__type_Frais kilométriques,0.5474
23,categories__type_Parking,0.1648
27,categories__type_Repas à emporter,0.0851
8,categories__type_Carburant,0.0557
218,numeriques__mois,0.0555


#### Principaux facteurs poussant vers `Approuvée`

,Feature,Contribution SHAP
173,categories__project_code_SANS_PROJET,-0.7228
216,numeriques__tax_rate,-0.4276
217,numeriques__annee,-0.3400
220,numeriques__jour_semaine,-0.3203
221,numeriques__trimestre,-0.1131


## 14. Sauvegarde de l'artefact et des métadonnées

In [21]:
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_PATH = MODELS_DIR / "expenseai_model.joblib"
METADATA_PATH = MODELS_DIR / "expenseai_model_metadata.json"

created_at = datetime.now(timezone.utc).isoformat()
model_version = datetime.now(timezone.utc).strftime("1.0.0-%Y%m%d%H%M%S")
calibration_metadata = SELECTED_CALIBRATION or "none"
score_name = "probability" if SELECTED_CALIBRATION is not None else "risk_score"

artifact = {
    "artifact_format_version": 1,
    "model": final_estimator,
    "threshold": PRODUCTION_THRESHOLD,
    "features": FEATURES,
    "model_version": model_version,
    "positive_class": 1,
    "positive_label": "Refusée",
    "calibration": calibration_metadata,
    "score_name": score_name,
    "created_at_utc": created_at,
}
joblib.dump(artifact, ARTIFACT_PATH)

metadata = {
    "model_name": "ExpenseAI Logistic Regression",
    "model_version": model_version,
    "created_at_utc": created_at,
    "positive_class": 1,
    "positive_label": "Refusée",
    "threshold": PRODUCTION_THRESHOLD,
    "threshold_rule": "F2 maximal sur probabilités OOF groupées du train",
    "calibration": calibration_metadata,
    "score_name": score_name,
    "features": FEATURES,
    "training_rows": int(len(X_train)),
    "training_positive_rows": int(y_train.sum()),
    "metric_primary": "average_precision",
    "test_metrics": {key: float(value) for key, value in test_metrics.items()},
    "confusion_matrix": {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    },
}
with METADATA_PATH.open("w", encoding="utf-8") as metadata_file:
    json.dump(metadata, metadata_file, ensure_ascii=False, indent=2)

# Validation immédiate de la sérialisation sur une donnée strictement synthétique.
loaded_artifact = joblib.load(ARTIFACT_PATH)
probability_before_save = synthetic_score
probability_after_reload = float(
    loaded_artifact["model"].predict_proba(synthetic_expense)[:, 1][0]
)
assert np.isclose(
    probability_before_save,
    probability_after_reload,
    rtol=1e-12,
    atol=1e-12,
)
assert loaded_artifact["features"] == FEATURES
assert float(loaded_artifact["threshold"]) == PRODUCTION_THRESHOLD

from ml.predict import predict_expense

api_prediction = predict_expense(
    synthetic_expense.iloc[0].to_dict(),
    artifact_path=ARTIFACT_PATH,
)
display(
    pd.Series(
        {
            "Artefact": str(ARTIFACT_PATH.relative_to(PROJECT_ROOT)),
            "Métadonnées": str(METADATA_PATH.relative_to(PROJECT_ROOT)),
            "Taille artefact (octets)": ARTIFACT_PATH.stat().st_size,
            "Score avant sauvegarde": probability_before_save,
            "Score après rechargement": probability_after_reload,
            "Prédiction API synthétique": api_prediction,
        },
        name="Validation de sauvegarde",
    ).to_frame()
)

,Validation de sauvegarde
Artefact,models/expenseai_model.joblib
Métadonnées,models/expenseai_model_metadata.json
Taille artefact (octets),237839
Score avant sauvegarde,0.0044
Score après rechargement,0.0044
Prédiction API synthétique,"{'predicted_target': 0, 'predicted_label': 'Ap..."


## Synthèse du modèle final

In [22]:
def metric_fr(value: float) -> str:
    """Formate une métrique selon la notation française."""
    return f"{value:.4f}".replace(".", ",")


top_shap_features = ", ".join(shap_global_importance.head(5)["Feature"])
summary = f"""
### Configuration verrouillée

- **Modèle :** Logistic Regression (`C=1`, `penalty="l2"`, `class_weight="balanced"`, `max_iter=2000`, `random_state=42`).
- **Calibration :** {selected_method_label}. {calibration_reason}
- **Seuil final :** {PRODUCTION_THRESHOLD:.2f}, choisi par F2 maximal sur les probabilités OOF groupées du train.
- **Résultats OOF au seuil :** precision {metric_fr(scenario_f2['Precision'])}, recall {metric_fr(scenario_f2['Recall'])}, F2 {metric_fr(scenario_f2['F2'])}, {int(scenario_f2['FP'])} FP et {int(scenario_f2['FN'])} FN.

### Évaluation finale sur le test de référence

- **PR-AUC :** {metric_fr(test_metrics['average_precision'])}
- **Recall Refusée :** {metric_fr(test_metrics['recall_refusee'])}
- **Precision Refusée :** {metric_fr(test_metrics['precision_refusee'])}
- **F2 :** {metric_fr(test_metrics['f2_refusee'])}
- **Brier Score :** {metric_fr(test_metrics['brier_score'])}
- **Matrice de confusion :** TN={tn}, FP={fp}, FN={fn}, TP={tp}.

### Interprétation et limites

Les cinq features transformées les plus importantes selon la moyenne de |SHAP| sont : **{top_shap_features}**. Elles sont associées aux décisions historiques du modèle ; elles ne prouvent aucune causalité. Avec seulement 23 refus dans le test, recall et precision restent sensibles à quelques lignes. La période historique d'environ un an ne garantit pas la stabilité future : suivi de dérive, calibration et performance seront nécessaires.

ExpenseAI estime un risque et signale des dépenses à contrôler. La validation ou le refus demeure une décision humaine.
"""
display(Markdown(summary))


### Configuration verrouillée

- **Modèle :** Logistic Regression (`C=1`, `penalty="l2"`, `class_weight="balanced"`, `max_iter=2000`, `random_state=42`).
- **Calibration :** sigmoid. Sigmoid améliore clairement au moins une métrique probabiliste, maintient l'ECE et la PR-AUC, et reste la solution la plus simple.
- **Seuil final :** 0.04, choisi par F2 maximal sur les probabilités OOF groupées du train.
- **Résultats OOF au seuil :** precision 0,0904, recall 0,4945, F2 0,2610, 453 FP et 46 FN.

### Évaluation finale sur le test de référence

- **PR-AUC :** 0,0858
- **Recall Refusée :** 0,4783
- **Precision Refusée :** 0,0846
- **F2 :** 0,2477
- **Brier Score :** 0,0159
- **Matrice de confusion :** TN=1272, FP=119, FN=12, TP=11.

### Interprétation et limites

Les cinq features transformées les plus importantes selon la moyenne de |SHAP| sont : **categories__type_Frais kilométriques, numeriques__mois, categories__project_code_SANS_PROJET, numeriques__annee, categories__type_Parking**. Elles sont associées aux décisions historiques du modèle ; elles ne prouvent aucune causalité. Avec seulement 23 refus dans le test, recall et precision restent sensibles à quelques lignes. La période historique d'environ un an ne garantit pas la stabilité future : suivi de dérive, calibration et performance seront nécessaires.

ExpenseAI estime un risque et signale des dépenses à contrôler. La validation ou le refus demeure une décision humaine.


In [23]:
assert MODEL_LOCKED
assert set(groups_train).isdisjoint(set(groups_test))
assert ARTIFACT_PATH.is_file()
assert METADATA_PATH.is_file()
assert np.isfinite(list(test_metrics.values())).all()
assert not any("expense_group" == feature for feature in FEATURES)

print("Modèle final sauvegardé")
print("Calibration :", selected_method_label)
print(f"Seuil final : {PRODUCTION_THRESHOLD:.2f}")
print(f"PR-AUC test : {test_metrics['average_precision']:.4f}")
print(f"Recall refus test : {test_metrics['recall_refusee']:.4f}")
print("Artefact : models/expenseai_model.joblib")
print("Métadonnées : models/expenseai_model_metadata.json")

Modèle final sauvegardé
Calibration : sigmoid
Seuil final : 0.04
PR-AUC test : 0.0858
Recall refus test : 0.4783
Artefact : models/expenseai_model.joblib
Métadonnées : models/expenseai_model_metadata.json
